# Notebook 4: Ablation Study
**Paper:** *Agents for Agents: An Interrogator-Based Secure Framework for Autonomous IoUT*

This notebook reproduces the qualitative ablation study from Section VI.E:

| Config | Components | Accuracy | PDR | Marginal Gain |
|---|---|---|---|---|
| A | Static Auth only | 72.5% | 79.4% | — |
| B | + Bayesian Trust Scoring | 86.1% | 86.7% | +13.6%, +7.3% |
| C | + Transformer + Blockchain | **94.2%** | **91.6%** | +8.1%, +4.9% |

Results from Table 3 of the paper — no new data fabricated.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'matplotlib'], check=True)
if not os.path.exists('IoUT-Interrogator-Framework'):
    subprocess.run(['git', 'clone', 'https://github.com/aliakarma/IoUT-Interrogator-Framework.git'], check=True)
os.chdir('IoUT-Interrogator-Framework')
sys.path.insert(0, '.')
print('Ready.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from analysis.plot_ablation import ABLATION_DATA, plot_ablation
print('Imports OK.')

In [ ]:
# ── Cell 3: Ablation table ─────────────────────────────────────────────────
df = pd.DataFrame({
    'Config':     ['A — Static Auth', 'B — + Bayesian Trust', 'C — + Transformer + BC'],
    'Accuracy (%)': ABLATION_DATA['accuracy'],
    'PDR (%)':      ABLATION_DATA['pdr'],
    'Acc Δ':        ['—', '+13.6%', '+8.1%'],
    'PDR Δ':        ['—', '+7.3%',  '+4.9%'],
})
print(df.to_string(index=False))

In [ ]:
# ── Cell 4: Ablation bar chart ─────────────────────────────────────────────
os.makedirs('analysis/plots', exist_ok=True)
plot_ablation('analysis/plots/ablation_study.png')
from IPython.display import Image
Image('analysis/plots/ablation_study.png')

In [ ]:
# ── Cell 5: Component attribution analysis ────────────────────────────────
configs    = ABLATION_DATA['configs']
accuracies = ABLATION_DATA['accuracy']
pdrs       = ABLATION_DATA['pdr']

print('=== Component Attribution ===')
print(f'\nA → B (adding Bayesian trust scoring):')
print(f'  Accuracy gain: +{accuracies[1]-accuracies[0]:.1f} pp  '
      f'({(accuracies[1]-accuracies[0])/accuracies[0]*100:.1f}% relative improvement)')
print(f'  PDR gain:      +{pdrs[1]-pdrs[0]:.1f} pp')
print(f'  Interpretation: Basic behavioral monitoring alone accounts for '
      f'{(accuracies[1]-accuracies[0])/(accuracies[2]-accuracies[0])*100:.0f}% '
      f'of total accuracy gain.')

print(f'\nB → C (adding transformer inference + blockchain governance):')
print(f'  Accuracy gain: +{accuracies[2]-accuracies[1]:.1f} pp  '
      f'({(accuracies[2]-accuracies[1])/accuracies[1]*100:.1f}% relative improvement)')
print(f'  PDR gain:      +{pdrs[2]-pdrs[1]:.1f} pp')
print(f'  Interpretation: Transformer + blockchain provides independent non-redundant')
print(f'  gain, validating the joint architectural design (see paper Section VI.E).')

print(f'\nTotal improvement (A → C):')
print(f'  Accuracy: +{accuracies[2]-accuracies[0]:.1f} pp ({(accuracies[2]-accuracies[0])/accuracies[0]*100:.1f}% relative)')
print(f'  PDR:      +{pdrs[2]-pdrs[0]:.1f} pp')

In [ ]:
# ── Cell 6: Marginal gains radar chart ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, metric, values, title, color in [
    (axes[0], 'Accuracy (%)', accuracies, 'Anomaly Detection Accuracy', '#2171b5'),
    (axes[1], 'PDR (%)',      pdrs,       'Packet Delivery Ratio',      '#2ca02c'),
]:
    bars = ax.barh(['Config A', 'Config B', 'Config C'], values,
                   color=[f'{color}66', f'{color}aa', color],
                   edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
    ax.set_xlabel(metric, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(60, 100)
    ax.grid(True, axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Ablation Study — Horizontal Performance View', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('analysis/plots/ablation_horizontal.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: analysis/plots/ablation_horizontal.png')